# Simple ML Questions: Ultralow Vaccine Distribution Data

This beginner notebook uses the Jian Sun ultralow-temperature dataset stored in Google Drive. It uses only `numpy`, `pandas`, `matplotlib`, linear regression, logistic regression, and K-means.

Important: this is a learning project. It detects patterns for review; it does **not** decide whether a vaccine is usable.

## What this dataset can answer

| Original question | What this notebook does |
|---|---|
| Recovery time from excursion severity | Uses temperature change as a simple proxy. Full recovery analysis needs real excursion and recovery labels. |
| Factors associated with temperature rise | Yes: uses time, mean temperature, sensor spread, O2, and CO2. |
| Can sensors validate one another? | Yes: uses one temperature channel to estimate another. |
| Normal vs investigation-needed? | Yes, using unusually rapid temperature change as a learning label. |
| Probability of a warm excursion | Yes, using the warmest 10% of observed windows as a relative label. |
| Possible freezing exposure | Not meaningful here without a product-specific threshold. |
| Open-door or handling event | Only a possible-event proxy; the dataset has no confirmed door-event labels. |

In [ ]:
# These are the only libraries used in this notebook.
import zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.cluster import KMeans

drive.mount('/content/drive')
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Load the data

This finds the existing ZIP file in Drive, extracts it, and loads the first temperature test. The code does not depend on a particular sensor brand.

In [ ]:
# Find the dataset ZIP in Google Drive.
drive_root = Path('/content/drive/MyDrive')
zip_files = list(drive_root.rglob('14888121*.zip'))
if not zip_files:
    raise FileNotFoundError('Could not find the dataset ZIP in Google Drive.')

extract_dir = Path('/content/ultralow_data')
extract_dir.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_files[0]) as z:
    z.extractall(extract_dir)

data_dir = next(extract_dir.rglob('14888121'))
raw = pd.read_csv(data_dir / 'Test1_TempCO2O2.csv', low_memory=False).iloc[2:].copy()
raw['hours'] = pd.to_timedelta(raw['Time Elapsed'].astype(str), errors='coerce').dt.total_seconds() / 3600

# Convert all possible measurement columns to numbers.
for column in raw.columns:
    if column not in ['date', 'time', 'Time Elapsed', 'hours']:
        raw[column] = pd.to_numeric(raw[column], errors='coerce')
raw = raw.dropna(subset=['hours'])
print('Rows loaded:', len(raw))

In [ ]:
# Find usable temperature columns. O2 and CO2 are kept as environmental features, not temperatures.
not_temperature = {'hours', 'O2', 'CO2', 'Ambient', 'Unnamed: 63', 'Unnamed: 64'}
temp_columns = [c for c in raw.columns if c not in not_temperature and pd.api.types.is_numeric_dtype(raw[c]) and raw[c].notna().sum() > 100]
print('Temperature channels found:', len(temp_columns))
print('First five channels:', temp_columns[:5])

# Make one simple table with the averages we will use.
data = raw[['hours', 'O2', 'CO2'] + temp_columns].copy()
data['mean_temperature'] = data[temp_columns].mean(axis=1)
data['sensor_spread'] = data[temp_columns].max(axis=1) - data[temp_columns].min(axis=1)
data = data.dropna(subset=['mean_temperature', 'sensor_spread', 'O2', 'CO2']).reset_index(drop=True)
data[['hours', 'mean_temperature', 'sensor_spread', 'O2', 'CO2']].head()

## 2. Basic visualization

The line is the average across all temperature channels. The shaded area shows how far apart the channels are. A wider area means more disagreement between sensors.

In [ ]:
# Show only the first 24 hours so the plot is easy to read.
first_day = data[data['hours'] <= 24]
plt.figure(figsize=(12,5))
plt.plot(first_day['hours'], first_day['mean_temperature'], color='tab:blue', label='Average temperature')
plt.fill_between(first_day['hours'], first_day['mean_temperature'] - first_day['sensor_spread']/2, first_day['mean_temperature'] + first_day['sensor_spread']/2, color='tab:blue', alpha=.2, label='Sensor disagreement')
plt.title('Temperature Pattern During the First 24 Hours')
plt.xlabel('Hours elapsed')
plt.ylabel('Recorded temperature (dataset units)')
plt.legend()
plt.show()

## 3. Linear regression: can one sensor estimate another?

**Question:** Can the existing temperature sensors help validate one another?

We use the first sensor to predict the second sensor. If the points are close to the dashed line, the two channels have a consistent relationship. Large gaps may be worth reviewing for placement or sensor disagreement.

In [ ]:
sensor_a, sensor_b = temp_columns[0], temp_columns[1]
sensor_data = data[[sensor_a, sensor_b]].dropna()
model_sensor = LinearRegression().fit(sensor_data[[sensor_a]], sensor_data[sensor_b])
predicted_b = model_sensor.predict(sensor_data[[sensor_a]])
r2 = model_sensor.score(sensor_data[[sensor_a]], sensor_data[sensor_b])
print(f'R-squared: {r2:.3f}')

plt.figure(figsize=(7,6))
plt.scatter(sensor_data[sensor_a], sensor_data[sensor_b], s=8, alpha=.35, label='Actual readings')
plt.plot(sensor_data[sensor_a], predicted_b, color='red', label='Linear regression line')
plt.title('Sensor Validation Check')
plt.xlabel(f'Sensor A: {sensor_a}')
plt.ylabel(f'Sensor B: {sensor_b}')
plt.legend()
plt.show()

## 4. Linear regression: what is associated with temperature change?

**Question:** Which available factors are associated with temperature change?

The target is the change in average temperature at the next reading. The bar chart compares the direction of each relationship: positive bars are associated with warming, while negative bars are associated with cooling. This shows association, not cause.

In [ ]:
# The shift(-1) makes the target the NEXT reading, not the current one.
data['next_temperature'] = data['mean_temperature'].shift(-1)
data['next_change'] = data['next_temperature'] - data['mean_temperature']
model_data = data.dropna(subset=['next_change']).copy()
features = ['hours', 'mean_temperature', 'sensor_spread', 'O2', 'CO2']
X = model_data[features]
y = model_data['next_change']
change_model = LinearRegression().fit(X, y)
print(f'R-squared: {change_model.score(X, y):.3f}')

plt.figure(figsize=(9,4))
plt.bar(features, change_model.coef_, color=['tab:blue' if x >= 0 else 'tab:orange' for x in change_model.coef_])
plt.axhline(0, color='black', linewidth=1)
plt.title('Linear Regression Coefficients for Next Temperature Change')
plt.ylabel('Coefficient (different units, so compare direction carefully)')
plt.xticks(rotation=20)
plt.show()

## 5. Logistic regression: normal or investigation-needed?

**Question:** Can the system classify whether the next reading is stable or needs investigation?

Because this dataset has no confirmed failure labels, we make a simple learning label: the 10% largest next-temperature changes are called `investigation-needed`. This does not mean a real failure happened.

In [ ]:
# Create a simple 0/1 label. 1 means the next change is unusually large for THIS dataset.
cutoff = model_data['next_change'].abs().quantile(.90)
model_data['investigation_needed'] = (model_data['next_change'].abs() >= cutoff).astype(int)

X = model_data[features]
y = model_data['investigation_needed']
investigation_model = LogisticRegression(max_iter=1000, class_weight='balanced').fit(X, y)
probability = investigation_model.predict_proba(X)[:, 1]
print('Investigation-needed windows:', y.sum(), 'out of', len(y))

plt.figure(figsize=(10,4))
plt.scatter(model_data['hours'], probability, c=y, cmap='coolwarm', s=12, alpha=.7)
plt.colorbar(label='Actual learning label: 0 = stable, 1 = investigation-needed')
plt.title('Probability That the Next Reading Needs Investigation')
plt.xlabel('Hours elapsed')
plt.ylabel('Predicted probability')
plt.ylim(-.05, 1.05)
plt.show()

## 6. Logistic regression: probability of a warm relative reading

**Question:** What is the probability that the next reading is relatively warm?

Here, `warm` means the next reading is in the warmest 10% of this dataset. It is not a manufacturer temperature limit.

In [ ]:
# Make a second 0/1 learning label based on the next temperature value.
warm_cutoff = model_data['next_temperature'].quantile(.90)
model_data['warm_relative'] = (model_data['next_temperature'] >= warm_cutoff).astype(int)
warm_model = LogisticRegression(max_iter=1000, class_weight='balanced').fit(model_data[features], model_data['warm_relative'])
warm_probability = warm_model.predict_proba(model_data[features])[:, 1]

plt.figure(figsize=(10,4))
plt.plot(model_data['hours'], warm_probability, color='tab:red')
plt.axhline(.5, color='black', linestyle='--', label='50% probability')
plt.title('Probability That the Next Reading Is Relatively Warm')
plt.xlabel('Hours elapsed')
plt.ylabel('Predicted probability')
plt.ylim(-.05, 1.05)
plt.legend()
plt.show()

## 7. K-means: find groups of similar temperature behavior

K-means does not predict a label. It groups readings that look similar. Here it creates three groups using average temperature and sensor disagreement. The colors show groups with similar patterns, which can help identify normal patterns and unusual groups to investigate.

In [ ]:
# K-means groups each row into one of three temperature-pattern clusters.
cluster_features = data[['mean_temperature', 'sensor_spread']].dropna().copy()
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(cluster_features)

plt.figure(figsize=(8,5))
plt.scatter(cluster_features['mean_temperature'], cluster_features['sensor_spread'], c=clusters, cmap='viridis', s=10, alpha=.6)
plt.scatter(kmeans.cluster_centers_[:,0], kmeans.cluster_centers_[:,1], c='red', marker='X', s=180, label='Cluster centers')
plt.title('K-means Groups of Similar Temperature Patterns')
plt.xlabel('Average temperature')
plt.ylabel('Sensor disagreement')
plt.legend()
plt.show()

print('Cluster centers:')
print(pd.DataFrame(kmeans.cluster_centers_, columns=['mean_temperature', 'sensor_spread']))

## Final takeaway

- Linear regression answers: **how much?** For example, how much one sensor changes with another.
- Logistic regression answers: **how likely?** For example, what is the probability of an unusual next reading.
- K-means answers: **which readings look similar?** It groups patterns without needing known labels.

For a real lab project, add verified temperature limits and confirmed event labels before using these models for operational decisions.